## data loading

In [1]:
import sys
import os


sys.path.append('..')
os.chdir('..')

In [ ]:
# dataset
import os
import torch

from starry.utils.config import Configuration
from starry.utils.dataset_factory import loadDataset


torch.set_printoptions(profile="full")

DATA_DIR = os.getenv('DATA_DIR')

config = Configuration.create('./configs/paraff-visionund-data.yaml', volatile=True)
data, = loadDataset(config, data_dir=DATA_DIR, splits='*0/1')

it = iter(data)
batch = next(it)
batch

/home/camus/work/deep-starry/env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
You are using the default legacy behaviour of the <class 'transformers.models.llama.tokenization_llama_fast.LlamaTokenizerFast'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565 - if you loaded a llama tokenizer from a GGUF file you can ignore this message.


In [ ]:
batch['input_ids'].shape, batch['target_mask'].shape, batch['img_emb'].shape, batch['image_seq_mask'].shape

(torch.Size([2, 765]),
 torch.Size([2, 765]),
 torch.Size([2, 1, 576, 1024]),
 torch.Size([2, 765]))

In [ ]:
print(data.dataset.tokenizer.decode(batch['input_ids'][0]))

<｜begin▁of▁sentence｜>You are a helpful language and vision assistant. You are able to understand the visual content that the user provides, and assist the user with a variety of tasks using natural language.

<|User|>: <begin_of_image><image_placeholder><image_placeholder><image_placeholder><image_placeholder><image_placeholder><image_placeholder><image_placeholder><image_placeholder><image_placeholder><image_placeholder><image_placeholder><image_placeholder><image_placeholder><image_placeholder><image_placeholder><image_placeholder><image_placeholder><image_placeholder><image_placeholder><image_placeholder><image_placeholder><image_placeholder><image_placeholder><image_placeholder><image_placeholder><image_placeholder><image_placeholder><image_placeholder><image_placeholder><image_placeholder><image_placeholder><image_placeholder><image_placeholder><image_placeholder><image_placeholder><image_placeholder><image_placeholder><image_placeholder><image_placeholder><image_placeholder><imag

---
## model

In [1]:
import sys
import os


sys.path.append('..')
os.chdir('..')

In [2]:
# dataset
import os
import torch

from starry.utils.config import Configuration
from starry.utils.dataset_factory import loadDataset
from starry.utils.model_factory import loadModel


torch.set_printoptions(profile="full")

DATA_DIR = os.getenv('DATA_DIR')

config = Configuration.create('./configs/paraff-visionund-test.yaml', volatile=True)
train, = loadDataset(config, data_dir=DATA_DIR, splits='*0/1', device='cuda')
model = loadModel(config['model'], postfix='Loss').cuda()

it = iter(train)

batch = next(it)
loss, metric = model(batch)

loss, metric

You are using the default legacy behaviour of the <class 'transformers.models.llama.tokenization_llama_fast.LlamaTokenizerFast'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565 - if you loaded a llama tokenizer from a GGUF file you can ignore this message.
2025-03-17 14:07:48.377841: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-03-17 14:07:48.377876: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-03-17 14:07:48

(tensor(12.0952, device='cuda:0', grad_fn=<NllLossBackward0>),
 {'loss': tensor(12.0952, device='cuda:0', grad_fn=<NllLossBackward0>),
  'acc': tensor(0., device='cuda:0')})

In [3]:
loss.backward()

In [ ]:
with torch.no_grad():
	inspection = model.inspectRun(batch)

inspection['target_flat'].shape, inspection['pred_flat'].shape

(torch.Size([224]), torch.Size([224, 102400]))

In [6]:
from transformers import AutoTokenizer


tokenizer = AutoTokenizer.from_pretrained("deepseek-ai/Janus-Pro-7B")
tokenizer

LlamaTokenizerFast(name_or_path='deepseek-ai/Janus-Pro-7B', vocab_size=100000, model_max_length=16384, is_fast=True, padding_side='left', truncation_side='right', special_tokens={'bos_token': '<｜begin▁of▁sentence｜>', 'eos_token': '<｜end▁of▁sentence｜>', 'pad_token': '<｜▁pad▁｜>', 'additional_special_tokens': ['<image_placeholder>', '<patch_placeholder>', '<|ref|>', '<|/ref|>', '<|det|>', '<|/det|>', '<|grounding|>', '<|User|>', '<|Assistant|>']}, clean_up_tokenization_spaces=False),  added_tokens_decoder={
	100000: AddedToken("<｜begin▁of▁sentence｜>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
	100001: AddedToken("<｜end▁of▁sentence｜>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
	100002: AddedToken("ø", rstrip=False, lstrip=False, single_word=False, normalized=True, special=False),
	100003: AddedToken("ö", rstrip=False, lstrip=False, single_word=False, normalized=True, special=False),
	100004: AddedToken("ú", rstrip=Fa

In [7]:
tokenizer.decode(inspection['target_flat'])

'OM K4 TN6 TD8 S1 Cg Md b Osup e g As D2 Dot EslurR VB S2 Cf Md e Osub D16 Bl b Osup D16 e D16 d As D16 c As D16 b D16 Br a D16 Bl g As D16 f As D16 e D16 d As D16 c As D16 Br EOM<｜end▁of▁sentence｜>OM K1 TN3 TD4 S1 Cg Md f As Osup D16 Bl g D16 f As D16 e D16 Br d D16 Bl f As D16 d D16 c As D16 Br b D16 Bl f Osup D16 b Osub D16 a D16 Br VB S2 Cf Mu c D8 Rest f As Osub D8 Bl b D8 c As D8 Br d D4 Etie VB S2 Cf Md a As Osub D4 b D4 a D8 Rest d D8 EOM<｜end▁of▁sentence｜>'

In [11]:
vocab = {}

for w, i in tokenizer.vocab.items():
	vocab[i] = w

vocab

{22560: 'Ġenabling',
 55580: 'telefono',
 90383: 'ĠMeditation',
 54861: 'Ð°Ð»Ñĥ',
 34851: 'ĠValues',
 52188: 'ĠAdvertising',
 36201: 'Ġconsola',
 70798: 'Ġcrater',
 29: '>',
 52394: 'appears',
 56501: 'endants',
 36395: 'çļĦé£İ',
 17428: 'release',
 31263: 'ĠCastella',
 80327: 'ĠInnoc',
 29661: 'Care',
 61314: 'çĢĳå¸ĥ',
 53121: 'Ġbathing',
 71840: 'Ġglaci',
 73939: 'Ġinteressant',
 55261: 'Ġmaritime',
 82799: 'Ġnozzle',
 23045: 'ç»´æĮģ',
 62439: 'Cher',
 26462: 'Ġproclam',
 12750: 'Ġqueries',
 93783: 'Ġselenium',
 18745: 'éķ¿çļĦ',
 97062: 'Ġhookup',
 8099: 'Ġtruly',
 39034: 'Ġwarmer',
 47248: 'etz',
 57445: 'Ġdumps',
 13108: 'Ġpassage',
 78379: 'herst',
 89311: 'Ġpedals',
 38676: 'ĠShield',
 55625: 'actus',
 24254: 'Ġpound',
 31657: 'åıĳçļĦ',
 62292: 'derabad',
 39371: 'ĠBackground',
 2779: 'ÑģÐ¸',
 25045: 'åľ°çļĦ',
 93789: 'ĠLlevant',
 61300: 'åİŁè°ħ',
 75034: 'Ġpruebas',
 4729: 'è¿ľ',
 2658: 'ĠPh',
 66158: 'IFF',
 7653: 'Ġdefinitely',
 84099: 'Ġtiresome',
 64518: 'ruly',
 15907: 'Ġvi

In [22]:
from starry.paraff.viewer import ParaffViewer


viewer = ParaffViewer(dict(_vocab=''))
viewer.vocab = vocab
viewer

In [ ]:
import matplotlib


matplotlib.use('WebAgg')


for k, v in inspection.items():
	inspection[k] = v.cpu()

inspection['target_flat'] = inspection['target_flat'].tolist()

In [ ]:
viewer.showBatch(None, inspection)

/home/camus/work/deep-starry/env/lib/python3.10/site-packages/tornado/websocket.py:631: UserWarning: Glyph 65372 (\N{FULLWIDTH VERTICAL LINE}) missing from font(s) DejaVu Sans.
  result = callback(*args, **kwargs)
